# Lab - Grounded Answers: No Source, No Answer

Three tasks that make ShopAssist answer only from what retrieval found, show where each answer came from, and route every answer without a source away from the customer.

| Task | What you learn |
| --- | --- |
| 1 | Retrieved sections as search_result blocks, with citations enabled |
| 2 | Read the answer and its sources; a question the policy does not cover cites nothing |
| 3 | No citation, no answer: a fallback, a review queue, and the grounding rate |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import math
import re
from collections import Counter
from pathlib import Path

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"
POLICY = Path("policy.txt").read_text()

SYSTEM = (
    "You are ShopAssist. Answer in one sentence, only from the "
    "search results. If they do not answer the question, say "
    "that the return policy does not cover it."
)

# The index from the chunking lesson: section chunks, ranked by keywords (BM25).
WORD = re.compile(r"[a-z0-9]+")


def words(text):
    return WORD.findall(text.lower())


class BM25:
    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks = chunks
        self.docs = [Counter(words(c["text"])) for c in chunks]
        self.avg = sum(sum(d.values()) for d in self.docs) / len(self.docs)
        df = Counter(w for d in self.docs for w in d)
        n = len(self.docs)
        self.idf = {w: math.log(1 + (n - f + 0.5) / (f + 0.5)) for w, f in df.items()}
        self.k1, self.b = k1, b

    def search(self, query, k=3):
        def score(doc):
            norm = self.k1 * (1 - self.b + self.b * sum(doc.values()) / self.avg)
            return sum(self.idf.get(w, 0) * doc[w] * (self.k1 + 1) / (doc[w] + norm)
                       for w in set(words(query)))
        ranked = sorted(zip(self.chunks, self.docs), key=lambda cd: -score(cd[1]))
        return [chunk for chunk, _ in ranked[:k]]


def chunk_by_section(text):
    chunks = []
    for block in text.split("\n\n"):
        lines = block.strip().split("\n")
        if len(lines) >= 2:
            chunks.append({"id": lines[0].title(), "text": block.strip()})
    return chunks


index = BM25(chunk_by_section(POLICY))

---

## Task 1 - Retrieved sections go in as search results

In [ ]:
# ============================================================
# TASK 1 - Retrieved sections go in as search results
# ============================================================
#
# WHAT TO DO
#   Write as_search_result(): each chunk becomes a search_result
#   block - a source (the file and the section), a title, the
#   text as its content, and citations enabled. ask() sends the
#   three best sections before the question. Ask about card
#   refunds.
#
# WHY IT MATTERS
#   An answer without a source cannot be checked, and an
#   assistant that invents a policy costs real money. Sent as
#   search results, the sections become sources the API can
#   point to: every claim in the answer carries a citation back
#   to the section it came from.
#
# WHERE TO SEE IT IN THE LECTURE
#   "That last field asks the API to attach citations to the
#   answer".
#
# HOW TO DO IT
#   Three blanks:
#
#       "policy.txt#" + chunk["id"]           the source: file and section
#       [{"type": "text", "text": chunk["text"]}]   the content, as text blocks
#       {"enabled": True}                     ask for citations
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def as_search_result(chunk):
        return {
            "type": "search_result",
            "source": ...,   # "policy.txt#" + chunk["id"]
            "title": chunk["id"],
            "content": ...,   # [{"type": "text", "text": chunk["text"]}]
            "citations": ...,   # {"enabled": True}
        }

    def ask(question):   # provided: search, then send the results before the question
        results = [as_search_result(chunk) for chunk in index.search(question)]
        return client.messages.create(
            model=MODEL, max_tokens=300, output_config={"effort": "low"}, system=SYSTEM,
            messages=[{"role": "user", "content": results + [{"type": "text", "text": question}]}])

    response = ask("How long does a card refund take?")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for block in response.content:
    print(repr(block.text), "| cited:", [c.source for c in block.citations or []])

check("cite_blocks", as_search_result=as_search_result, response=response)

---

## Task 2 - Read the answer and where it came from

In [ ]:
# ============================================================
# TASK 2 - Read the answer and where it came from
# ============================================================
#
# WHAT TO DO
#   Write read(): a cited answer comes back as several text
#   blocks, so join their text, and collect the sources from
#   their citations. Ask two questions - one the policy
#   answers, one it does not.
#
# WHY IT MATTERS
#   The ranker always returns three sections, even for a
#   question about shipping to Canada. Retrieval only decides
#   what Claude reads; the instruction to answer only from the
#   results is what makes Claude say "not covered" - and then
#   nothing is cited.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Claude read them, found no answer, and said so. Nothing was
#   cited".
#
# HOW TO DO IT
#   Three blanks:
#
#       block.text                       every text block, joined
#       block.citations or []            a block may carry no citations
#       c.source                         where the claim came from
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def read(response):
        answer, sources = "", set()
        for block in response.content:
            if block.type == "text":
                answer += ...   # block.text
                sources |= {... for c in ...}   # c.source, block.citations or []
        return answer.strip(), sorted(sources)

    answers = {}
    for question in ["How long does a card refund take?", "Do you ship to Canada?"]:
        answers[question] = read(ask(question))
        print("Q:", question)
        print("A:", answers[question][0])
        print("   cited:", ", ".join(answers[question][1]) or "nothing")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Task 1 included."
                     .format(exc)) from None

check("cite_sources", answers=answers)

---

## Task 3 - An answer without a citation is unverified

In [ ]:
# ============================================================
# TASK 3 - An answer without a citation is unverified
# ============================================================
#
# WHAT TO DO
#   Write deliver(): a cited answer goes to the customer; an
#   answer with no citation goes to a person, and the customer
#   gets the fallback. Run a batch of five questions and report
#   the grounding rate - the share answered with a source.
#
# WHY IT MATTERS
#   In production you cannot read every answer. A rule in code
#   decides: no source, no answer - the case goes to a fallback
#   or a human. The grounding rate is the number to watch: when
#   it drops, customers are asking what your documents do not
#   cover, and the content needs work, not the model.
#
# WHERE TO SEE IT IN THE LECTURE
#   "In production, an answer with no citation can go to a
#   fallback or to a human".
#
# HOW TO DO IT
#   Three blanks:
#
#       if sources:                 cited: it may go out
#       REVIEW.append(question)     no source: a person looks at it
#       answered / len(results)     the share grounded in a source
# ============================================================

FALLBACK = "Thanks for your question - a colleague will reply within one business day."
REVIEW = []   # provided: the questions a person has to answer
BATCH = ["How long does a card refund take?", "Can I return earrings?",
         "Do you ship to Canada?", "Who approves a refund of 250?", "Can I pay with crypto?"]

try:
    # TODO: replace each ... below with the value named beside it
    def deliver(question):
        answer, sources = read(ask(question))
        if ...:   # sources
            return {"status": "answered", "answer": answer, "sources": sources}
        ...   # REVIEW.append(question)
        return {"status": "needs_review", "answer": FALLBACK, "sources": []}
    
    results = {q: deliver(q) for q in BATCH}
    answered = sum(r["status"] == "answered" for r in results.values())
    grounding_rate = ...   # answered / len(results)
    if grounding_rate is ... or ... in REVIEW:   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Tasks 1 to 2 included."
                     .format(exc)) from None
for question, r in results.items():
    print(f"{r['status']:13} {question}  {r['sources']}")
print(f"grounding rate {grounding_rate:.0%}; to review: {REVIEW}")

check("cite_gate", results=results, review=REVIEW, rate=grounding_rate, fallback=FALLBACK)

---

## Done

Send retrieved text as search results with citations, tell Claude to answer only from them and to say when they do not cover the question, and treat an answer without a citation as unverified - a fallback or a person, never the customer.